# Лабораторная работа 10. Стратегии декодирования и KV-cache (студенческая версия)

**Курс:** Методы обработки естественного языка · **Блок 4:** инференс и эффективность

**Цель.** Понять, как устроена генерация текста, где тратятся время и память, и как ограничить выход модели формальной грамматикой.

**Что нужно сдать:**
1. Greedy, beam search, температуру, top-k, top-p, min-p — для собственной модели и для предобученной LLM.
2. Сравнение стратегий по разнообразию (distinct-n, self-BLEU), повторяемости и perplexity.
3. KV-cache для mini-GPT: совпадение с генерацией без кэша, ускорение, рост памяти.
4. Перевод MHA в GQA усреднением голов K/V: потеря качества и экономия памяти.
5. Ограниченную генерацию по регулярному выражению маскированием логитов.

**Теория.**
- **Top-p (nucleus):** выбирается минимальное множество самых вероятных токенов с суммарной вероятностью \(\ge p\).
- **Min-p:** остаются токены с \(p_i \ge p_{min}\cdot\max_j p_j\) — порог адаптируется к уверенности модели.
- **Память KV-cache:** \(M = 2 \cdot L \cdot T \cdot n_{kv} \cdot d_{head} \cdot b\) байт на последовательность (\(b\) — байт на число). В GQA \(n_{kv} = H/g\).
- **Фаза decode** memory-bound: на каждый токен читаются все веса и весь кэш, а вычислений мало. Поэтому latency оценивается как (байты весов + байты кэша) / пропускная способность памяти.

In [ ]:
# ==== Общая настройка (одинакова во всех работах) ====
import os, math, time, json, random, pathlib, urllib.request
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

FAST_DEV_RUN = True          # True — сокращённый прогон (CPU, минуты); False — полный эксперимент
SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
ARTIFACTS = pathlib.Path("artifacts"); ARTIFACTS.mkdir(exist_ok=True)
DATA = pathlib.Path("data"); DATA.mkdir(exist_ok=True)

def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
set_seed()

def download(url, name):
    path = DATA / name
    if not path.exists():
        urllib.request.urlretrieve(url, path)
    return path

def load_shakespeare():
    url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
    return download(url, "shakespeare.txt").read_text(encoding="utf-8")

def load_russian():
    """Русский корпус: «Война и мир» (Л. Н. Толстой), т. 1, из открытого репозитория.
    При недоступности сети используется встроенный фрагмент."""
    urls = ["https://raw.githubusercontent.com/mmcdermott/nlp-war-and-peace/master/data/war_and_peace_ru.txt",
            "https://raw.githubusercontent.com/nevmenandr/word2vec-russian-novels/master/books_before/WarAndPeace.txt"]
    for i, u in enumerate(urls):
        try:
            t = download(u, f"ru_{i}.txt").read_text(encoding="utf-8", errors="ignore")
            if len(t) > 100_000: return t
        except Exception:
            pass
    base = ("Все счастливые семьи похожи друг на друга, каждая несчастливая семья несчастлива по-своему. "
            "Всё смешалось в доме Облонских. Жена узнала, что муж был в связи с бывшею в их доме француженкою-гувернанткой, "
            "и объявила мужу, что не может жить с ним в одном доме. ")
    return base * 400

print(f"torch {torch.__version__}, device={DEVICE}, FAST_DEV_RUN={FAST_DEV_RUN}")

In [ ]:
# ==== Эталонный mini-GPT (решение ЛР 6), используется как готовый модуль ====
from dataclasses import dataclass, asdict

class CharTokenizer:
    def __init__(self, text):
        self.chars = sorted(set(text)); self.stoi = {c: i for i, c in enumerate(self.chars)}
        self.vocab_size = len(self.chars)
    def encode(self, s): return [self.stoi[c] for c in s if c in self.stoi]
    def decode(self, ids): return "".join(self.chars[i] for i in ids)

@dataclass
class GPTConfig:
    vocab_size: int = 65
    block_size: int = 128
    n_layer: int = 4
    n_head: int = 4
    n_embd: int = 128
    dropout: float = 0.0

class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-6):
        super().__init__(); self.eps = eps; self.weight = nn.Parameter(torch.ones(d))
    def forward(self, x):
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps) * self.weight

def rope_cache(T, d_head, base=10000.0, device="cpu", scale=1.0):
    inv = 1.0 / (base ** (torch.arange(0, d_head, 2, device=device).float() / d_head))
    t = torch.arange(T, device=device).float() / scale
    ang = torch.outer(t, inv)                        # (T, d_head/2)
    return ang.cos(), ang.sin()

def apply_rope(x, cos, sin):                         # x: (B, H, T, d_head)
    x1, x2 = x[..., 0::2], x[..., 1::2]
    out = torch.stack([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1)
    return out.flatten(-2)

class CausalSelfAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.n_head, self.d_head = cfg.n_head, cfg.n_embd // cfg.n_head
        self.qkv = nn.Linear(cfg.n_embd, 3 * cfg.n_embd, bias=False)
        self.proj = nn.Linear(cfg.n_embd, cfg.n_embd, bias=False)
    def forward(self, x, cos, sin):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        q, k, v = (t.view(B, T, self.n_head, self.d_head).transpose(1, 2) for t in (q, k, v))
        q, k = apply_rope(q, cos[:T], sin[:T]), apply_rope(k, cos[:T], sin[:T])
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        return self.proj(y.transpose(1, 2).reshape(B, T, C))

class SwiGLU(nn.Module):
    def __init__(self, d, hidden):
        super().__init__()
        self.w1, self.w3 = nn.Linear(d, hidden, bias=False), nn.Linear(d, hidden, bias=False)
        self.w2 = nn.Linear(hidden, d, bias=False)
    def forward(self, x): return self.w2(F.silu(self.w1(x)) * self.w3(x))

class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.n1, self.attn = RMSNorm(cfg.n_embd), CausalSelfAttention(cfg)
        self.n2, self.mlp = RMSNorm(cfg.n_embd), SwiGLU(cfg.n_embd, int(8 * cfg.n_embd / 3))
    def forward(self, x, cos, sin):
        x = x + self.attn(self.n1(x), cos, sin)
        return x + self.mlp(self.n2(x))

class GPT(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.wte = nn.Embedding(cfg.vocab_size, cfg.n_embd)
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layer)])
        self.norm = RMSNorm(cfg.n_embd)
        self.head = nn.Linear(cfg.n_embd, cfg.vocab_size, bias=False)
        self.head.weight = self.wte.weight                    # weight tying
        cos, sin = rope_cache(4 * cfg.block_size, cfg.n_embd // cfg.n_head)
        self.register_buffer("cos", cos, persistent=False); self.register_buffer("sin", sin, persistent=False)
        self.apply(self._init)
        for n, p in self.named_parameters():                  # масштабирование выходных проекций по глубине
            if n.endswith("proj.weight") or n.endswith("w2.weight"):
                nn.init.normal_(p, 0.0, 0.02 / math.sqrt(2 * cfg.n_layer))
    def _init(self, m):
        if isinstance(m, (nn.Linear, nn.Embedding)): nn.init.normal_(m.weight, 0.0, 0.02)
    def forward(self, idx, targets=None):
        x = self.wte(idx)
        for b in self.blocks: x = b(x, self.cos, self.sin)
        logits = self.head(self.norm(x))
        loss = None if targets is None else F.cross_entropy(logits.flatten(0, 1), targets.flatten())
        return logits, loss
    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0):
        for _ in range(max_new_tokens):
            logits, _ = self(idx[:, -self.cfg.block_size:])
            probs = F.softmax(logits[:, -1] / temperature, dim=-1)
            idx = torch.cat([idx, torch.multinomial(probs, 1)], dim=1)
        return idx

def n_params(model):
    return sum(p.numel() for p in model.parameters())

def get_batch(data, block_size, batch_size, device=DEVICE):
    ix = torch.randint(len(data) - block_size - 1, (batch_size,))
    x = torch.stack([data[i:i + block_size] for i in ix]); y = torch.stack([data[i + 1:i + 1 + block_size] for i in ix])
    return x.to(device), y.to(device)

@torch.no_grad()
def estimate_loss(model, data, block_size, batch_size=32, iters=20):
    model.eval(); losses = [model(*get_batch(data, block_size, batch_size))[1].item() for _ in range(iters)]
    model.train(); return float(np.mean(losses))

def train_lm(model, train_data, val_data, steps, lr=3e-3, batch_size=32, warmup=50, log_every=100):
    opt = torch.optim.AdamW(model.parameters(), lr=lr, betas=(0.9, 0.95), weight_decay=0.1)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1, (s + 1) / warmup) * 0.5 * (1 + math.cos(math.pi * min(s, steps) / steps)))
    hist = []
    for s in range(steps):
        x, y = get_batch(train_data, model.cfg.block_size, batch_size)
        _, loss = model(x, y); opt.zero_grad(set_to_none=True); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step(); sched.step()
        if s % log_every == 0 or s == steps - 1:
            hist.append((s, loss.item(), estimate_loss(model, val_data, model.cfg.block_size)))
    return hist

def get_reference_gpt(text, cfg=None, steps=None, name=None):
    """Загружает или обучает эталонную символьную модель на тексте."""
    tok = CharTokenizer(text)
    data = torch.tensor(tok.encode(text), dtype=torch.long)
    n = int(0.9 * len(data)); train_data, val_data = data[:n], data[n:]
    cfg = cfg or (GPTConfig(tok.vocab_size, 64, 2, 4, 64) if FAST_DEV_RUN else GPTConfig(tok.vocab_size, 256, 6, 6, 384))
    name = name or f"ref_gpt_L{cfg.n_layer}_d{cfg.n_embd}_T{cfg.block_size}.pt"
    path = ARTIFACTS / name
    model = GPT(cfg).to(DEVICE)
    if path.exists():
        model.load_state_dict(torch.load(path, map_location=DEVICE))
    else:
        train_lm(model, train_data, val_data, steps or (300 if FAST_DEV_RUN else 5000))
        torch.save(model.state_dict(), path)
    return model, tok, train_data, val_data

In [ ]:
# ==== Загрузка предобученных моделей Hugging Face ====
from transformers import AutoTokenizer, AutoModelForCausalLM

def load_lm(name, dtype=None):
    tok = AutoTokenizer.from_pretrained(name)
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    dtype = dtype or (torch.float16 if DEVICE == "cuda" else torch.float32)
    try:
        model = AutoModelForCausalLM.from_pretrained(name, dtype=dtype)
    except TypeError:
        model = AutoModelForCausalLM.from_pretrained(name, torch_dtype=dtype)
    return model.to(DEVICE).eval(), tok

@torch.no_grad()
def perplexity(model, tok, text, max_len=512, stride=256, max_windows=None):
    ids = tok(text, return_tensors="pt").input_ids.to(DEVICE)
    nll, cnt, prev = 0.0, 0, 0
    for i, start in enumerate(range(0, ids.size(1) - 1, stride)):
        if max_windows and i >= max_windows: break
        end = min(start + max_len, ids.size(1)); x = ids[:, start:end]
        tgt = x.clone(); tgt[:, : max(0, prev - start)] = -100
        out = model(x, labels=tgt)
        n = (tgt[:, 1:] != -100).sum().item(); nll += out.loss.float().item() * n; cnt += n; prev = end
        if end == ids.size(1): break
    return math.exp(nll / cnt)

In [ ]:
text = load_shakespeare()
cfg = GPTConfig(65, 256, 2, 4, 128) if FAST_DEV_RUN else GPTConfig(65, 512, 6, 6, 384)
gpt, ctok, train_data, val_data = get_reference_gpt(text, cfg=cfg, steps=400 if FAST_DEV_RUN else 5000)
gpt.eval()
LLM_NAME = "HuggingFaceTB/SmolLM2-135M" if FAST_DEV_RUN else "Qwen/Qwen2.5-0.5B"
llm, llm_tok = load_lm(LLM_NAME)
print("mini-GPT:", n_params(gpt), "параметров;", LLM_NAME, n_params(llm))

## 1. Стратегии декодирования

Функция `filter_logits` получает логиты последнего шага `(B, V)` и возвращает отфильтрованные логиты (запрещённые — \(-\infty\)). Генерация работает с любой моделью через функцию `logits_fn(ids) -> (B, T, V)`.

In [ ]:
def filter_logits(logits, temperature=1.0, top_k=None, top_p=None, min_p=None):
    logits = logits / max(temperature, 1e-8)
    # TODO: ваш код
    raise NotImplementedError

# тесты
l = torch.log(torch.tensor([[0.5, 0.3, 0.15, 0.05]]))
assert torch.isfinite(filter_logits(l, top_k=2)).sum() == 2
assert torch.isfinite(filter_logits(l, top_p=0.7)).sum() == 2      # 0.5 + 0.3 ≥ 0.7
assert torch.isfinite(filter_logits(l, top_p=0.85)).sum() == 3
assert torch.isfinite(filter_logits(l, min_p=0.5)).sum() == 2      # порог 0.25
print("тесты фильтрации пройдены")

@torch.no_grad()
def sample(logits_fn, ids, n_new, greedy=False, **kw):
    for _ in range(n_new):
        logits = logits_fn(ids)[:, -1].float()
        nxt = logits.argmax(-1, keepdim=True) if greedy else torch.multinomial(filter_logits(logits, **kw).softmax(-1), 1)
        ids = torch.cat([ids, nxt], 1)
    return ids

@torch.no_grad()
def beam_search(logits_fn, ids, n_new, beam=4, length_penalty=1.0):
    """Beam search для одного префикса ids (1, T). Возвращает лучшую последовательность."""
    # TODO: ваш код
    raise NotImplementedError

gpt_fn = lambda ids: gpt(ids[:, -gpt.cfg.block_size:])[0]
llm_fn = lambda ids: llm(ids).logits
prompt = ctok.encode("ROMEO:\n")
for name, kw in [("greedy", dict(greedy=True)), ("T=0.7", dict(temperature=0.7)), ("top-p 0.9", dict(top_p=0.9)), ("min-p 0.1", dict(min_p=0.1))]:
    out = sample(gpt_fn, torch.tensor([prompt], device=DEVICE), 150, **kw)
    print(f"--- {name}\n{ctok.decode(out[0].tolist())}\n")
print("--- beam=4\n", ctok.decode(beam_search(gpt_fn, torch.tensor([prompt], device=DEVICE), 80)[0].tolist()))

## 2. Сравнение стратегий на предобученной LLM

Для 10–20 промптов сгенерируйте по 4 продолжения длиной 60 токенов каждой стратегией и посчитайте:
- **distinct-n** (\(n = 1, 2, 3\)) — доля уникальных n-грамм среди всех n-грамм продолжений;
- **self-BLEU** — средний BLEU каждого продолжения относительно остальных продолжений того же промпта (выше — меньше разнообразия);
- **повторяемость** — доля токенов, входящих в 4-грамму, уже встречавшуюся раньше в том же продолжении;
- **perplexity** продолжения под самой моделью (в полном режиме — под более крупной моделью того же семейства).

In [ ]:
PROMPTS = ["The history of the city begins", "In a recent study, scientists found that", "My favourite recipe for dinner is",
           "The main problem with modern education is", "Once upon a time, in a small village,", "The stock market today",
           "To install the package, first", "The most important lesson I learned", "During the night, the storm",
           "The new smartphone features"][: 4 if FAST_DEV_RUN else 10]
N_SAMPLES, GEN_LEN = (3, 40) if FAST_DEV_RUN else (4, 60)

def ngrams(seq, n): return [tuple(seq[i:i + n]) for i in range(len(seq) - n + 1)]

def bleu(hyp, refs, max_n=4):
    # TODO: ваш код (модифицированная точность n-грамм с клиппингом по максимуму в референсах, сглаживание +1, brevity penalty)
    raise NotImplementedError

def repetition_rate(seq, n=4):
    seen, rep = set(), 0
    for i, g in enumerate(ngrams(seq, n)):
        if g in seen: rep += 1
        seen.add(g)
    return rep / max(1, len(seq) - n + 1)

from collections import Counter
@torch.no_grad()
def self_ppl(prompt_ids, full_ids):
    logits = llm(full_ids).logits[0, len(prompt_ids) - 1:-1].float()
    return math.exp(F.cross_entropy(logits, full_ids[0, len(prompt_ids):]).item())

STRATEGIES = {"greedy": dict(greedy=True), "beam=4": "beam", "T=0.7": dict(temperature=0.7), "T=1.0": dict(temperature=1.0),
              "T=1.5": dict(temperature=1.5), "top-k 40": dict(top_k=40), "top-p 0.9": dict(top_p=0.9), "min-p 0.1": dict(min_p=0.1)}
# TODO: ваш код (сгенерируйте продолжения для всех стратегий и сведите метрики в таблицу)

## 3. KV-cache для mini-GPT

Реализуйте прямой проход эталонной модели с кэшем: для каждого слоя хранятся \(K, V\) формы `(B, H, T_past, d_head)`. На шаге decode вычисляется \(q, k, v\) только для нового токена; RoPE применяется с учётом смещения позиции.

1. Проверьте, что логиты с кэшем совпадают с логитами полного прохода (допуск 1e-4).
2. Измерьте время генерации 200 токенов с кэшем и без при разной длине промпта.
3. Сравните фактический объём кэша с формулой.

In [ ]:
@torch.no_grad()
def forward_cached(model, idx, cache, pos):
    """idx: (B, T_new); cache: список [(k, v)] по слоям или None; pos: позиция первого нового токена."""
    x = model.wte(idx); B, T, C = x.shape; new_cache = []
    for li, b in enumerate(model.blocks):
        a = b.attn; h = b.n1(x)
        # TODO: ваш код (q,k,v для новых токенов; RoPE с cos[pos:pos+T]; конкатенация с кэшем; каузальная маска при T > 1)
        raise NotImplementedError
        x = x + b.mlp(b.n2(x))
    return model.head(model.norm(x)), new_cache

@torch.no_grad()
def generate_cached(model, idx, n_new):
    logits, cache = forward_cached(model, idx, None, 0); pos = idx.size(1)
    for _ in range(n_new):
        nxt = logits[:, -1].argmax(-1, keepdim=True); idx = torch.cat([idx, nxt], 1)
        logits, cache = forward_cached(model, nxt, cache, pos); pos += 1
    return idx, cache

ids = torch.tensor([ctok.encode(text[:50])], device=DEVICE)
full = gpt(ids)[0]
part, cache = forward_cached(gpt, ids[:, :30], None, 0)
lg, cache = forward_cached(gpt, ids[:, 30:31], cache, 30)
for t in range(31, 50): lg2, cache = forward_cached(gpt, ids[:, t:t + 1], cache, t)
print("max |Δ| prefill:", (part - full[:, :30]).abs().max().item(), " шаг 30:", (lg[:, 0] - full[:, 30]).abs().max().item(),
      " шаг 49:", (lg2[:, 0] - full[:, 49]).abs().max().item())
assert torch.allclose(lg2[:, 0], full[:, 49], atol=1e-4)
g_cache, _ = generate_cached(gpt, ids[:, :20], 30); g_plain = sample(gpt_fn, ids[:, :20], 30, greedy=True)
assert torch.equal(g_cache, g_plain); print("greedy-генерация с кэшем совпадает с генерацией без кэша")

In [ ]:
# TODO: ваш код (время генерации с кэшем и без при разных длинах промпта; память кэша: факт и формула)

### Память KV-cache для реальных моделей

Посчитайте по формуле объём KV-cache для контекста 8k и 128k токенов в BF16 для моделей с конфигурацией, взятой из `config.json` (число слоёв, `num_key_value_heads`, `head_dim`). Сравните с объёмом весов.

In [ ]:
from transformers import AutoConfig
def kv_bytes(cfg_hf, T, bytes_per=2):
    # TODO: ваш код
    raise NotImplementedError
for name in ["HuggingFaceTB/SmolLM2-135M", "Qwen/Qwen2.5-0.5B", "Qwen/Qwen2.5-7B"]:
    c = AutoConfig.from_pretrained(name)
    print(f"{name:28s} heads={c.num_attention_heads}, kv_heads={c.num_key_value_heads}:  8k → {kv_bytes(c, 8192) / 2**30:.2f} ГиБ, 128k → {kv_bytes(c, 131072) / 2**30:.2f} ГиБ")

## 4. От MHA к GQA

Преобразуйте mini-GPT в GQA с группами размера \(g \in \{1, 2, 4\}\) (\(g\) голов запросов на одну пару K/V), усреднив веса \(W_K, W_V\) внутри группы (Ainslie et al., 2023). Реализуйте это заменой строк весов в `qkv` на средние по группе — функционально это эквивалентно GQA с повторением K/V-голов. Измерьте val loss до и после и после короткого дообучения (uptraining, около 5% шагов).

In [ ]:
import copy
def to_gqa(model, g):
    m = copy.deepcopy(model); C = m.cfg.n_embd; H = m.cfg.n_head; dh = C // H
    # TODO: ваш код (веса qkv: строки [C:2C] — K, [2C:3C] — V; головы по dh строк; усредните внутри групп по g голов)
    raise NotImplementedError
    return m

rows = []
for g in [1, 2, 4]:
    if gpt.cfg.n_head % g: continue
    m = to_gqa(gpt, g); before = estimate_loss(m, val_data, gpt.cfg.block_size, iters=20)
    m.train(); train_lm(m, train_data, val_data, 20 if FAST_DEV_RUN else 250, lr=1e-3, warmup=5, log_every=10 ** 9)
    after = estimate_loss(m, val_data, gpt.cfg.block_size, iters=20)
    rows.append({"g": g, "kv-голов": gpt.cfg.n_head // g, "кэш относительно MHA": 1 / g, "val до": before, "val после uptraining": after})
pd.DataFrame(rows).round(3)

## 5. Ограниченная генерация по регулярному выражению

Для символьной модели на каждом шаге оставьте только символы \(c\), для которых `prefix + c` остаётся допустимым префиксом строки, соответствующей шаблону (используйте `regex.fullmatch(pattern, s, partial=True)`). Генерация завершается, когда строка полностью соответствует шаблону и дальнейшее продолжение невозможно, либо по лимиту длины.

Шаблоны: дата `\d{2}\.\d{2}\.\d{4}`; JSON-подобная запись `\{"name": "[A-Za-z ]{2,12}", "age": [1-9][0-9]?\}`.

In [ ]:
import regex

@torch.no_grad()
def constrained_generate(model, prompt, pattern, max_len=60, temperature=0.8):
    ids = torch.tensor([ctok.encode(prompt)], device=DEVICE); out = ""
    # TODO: ваш код (маска допустимых символов → -inf для остальных; стоп, если допустимых нет или fullmatch без partial)
    raise NotImplementedError
    return out

# Шаблоны должны использовать только символы словаря Tiny Shakespeare (в нём нет, например, «{» и «"»)
for pat in [r"\d{2}\.\d{2}\.\d{4}", r"[A-Z]{3,10}:\n[A-Z][a-z]+( [a-z]+){2,5}[.!?]"]:
    assert all(c in ctok.stoi for c in "0123456789.:!? \n"), "символы шаблона отсутствуют в словаре"
    for s in range(3):
        set_seed(s); r = constrained_generate(gpt, "ROMEO:\n", pat)
        print(f"{r!r:55s} соответствует шаблону: {regex.fullmatch(pat, r) is not None}")

## Вопросы

1. Почему beam search даёт вырожденные и повторяющиеся тексты в открытой генерации, но хорошо работает в машинном переводе?
2. Сравните top-p и min-p при «уверенном» и «неуверенном» распределениях. В каком случае min-p оставляет больше кандидатов?
3. Оцените latency одного шага decode для 7B-модели в BF16 на GPU с пропускной способностью памяти 1 ТБ/с при батче 1. Что изменится при батче 64 и контексте 32k?
4. Почему GQA почти не ухудшает качество, но сильно уменьшает кэш? Как это влияет на арифметическую интенсивность decode?
5. Как обобщить ограниченную генерацию на BPE-токенизатор? Почему наивная проверка всех токенов словаря на каждом шаге слишком дорога и как её ускоряют (Outlines, автоматы над словарём)?